In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
import torch

INPUT_PATH = "/content/drive/MyDrive/DBAASP_Bioinformatics"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device   : {device}")

Device   : cuda


In [ ]:
import os, sys, shutil, math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.autograd import Variable
from transformers.models.bert.modeling_bert import BertEncoder, BertConfig

# Setup paths
INPUT_PATH   = "/content/drive/MyDrive/DBAASP_Bioinformatics"
N_SAMPLES    = 35000
DIFF_STEPS   = 500
BATCH_SIZE   = 64
LATENT_NOISE = 0.5

VAE_WEIGHTS  = f"{INPUT_PATH}/vae_weights.pth"
DIFF_WEIGHTS = f"{INPUT_PATH}/checkpoints_diffusion/phase2_final/best_model.pth"
ORG_MAP_PATH = f"{INPUT_PATH}/latent_/cond_latents_new/epidermidis.npy"
OUT_DIR      = f"{INPUT_PATH}/generatedPeptidesEpidermidis"

ENTROPY_DROP_THRESHOLD = 0.4
ENTROPY_WINDOW         = 5
MIN_SEQ_LEN            = 4

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
os.makedirs(OUT_DIR, exist_ok=True)

for fname in ["TransVAE.py", "peptide_vocab.pkl", "peptide_weight.npy"]:
    dst = f"/content/{fname}"
    if not os.path.exists(dst):
        shutil.copy(f"{INPUT_PATH}/{fname}", dst)
if "/content" not in sys.path:
    sys.path.insert(0, "/content")

from TransVAE import create_VAE, w2i, org_dict, params, subsequent_mask

# Vocab configurations
START_ID = w2i["<start>"]
EOS_ID   = w2i["<end>"]
PAD_ID   = w2i["_"]

def id_to_char(fid):
    return org_dict.get(fid - 1, "")

assert id_to_char(EOS_ID) == "<end>", f"EOS mismatch: {id_to_char(EOS_ID)}"
assert id_to_char(PAD_ID) == "_",     f"PAD mismatch: {id_to_char(PAD_ID)}"
print(f"Vocab OK: START={START_ID}  EOS={EOS_ID}  PAD={PAD_ID}\n")


def timestep_embedding(timesteps, dim, max_period=10000):
    half  = dim // 2
    freqs = torch.exp(
        -math.log(max_period)
        * torch.arange(start=0, end=half, dtype=torch.float32) / half
    ).to(device=timesteps.device)
    args      = timesteps[:, None].float() * freqs[None]
    embedding = torch.cat([torch.cos(args), torch.sin(args)], dim=-1)
    if dim % 2:
        embedding = torch.cat([embedding, torch.zeros_like(embedding[:, :1])], dim=-1)
    return embedding


class DModel(nn.Module):
    def __init__(self, num_class=2, max_length=1):
        super().__init__()
        self.input_channels = 128
        self.model_channels = 127
        self.out_channels   = 128
        self.num_class      = num_class
        self.drop_out       = 0.1
        self.max_length     = max_length

        config = BertConfig()
        config.hidden_dropout_prob      = self.drop_out
        config.max_position_embeddings  = self.max_length
        config.num_attention_heads      = 6
        config.num_hidden_layers        = 3
        config._attn_implementation     = "eager"

        self.control_embedding   = nn.Embedding(self.num_class, config.hidden_size)
        self.time_embed = nn.Sequential(
            nn.Linear(self.model_channels, config.hidden_size),
            nn.SiLU(),
            nn.Linear(config.hidden_size, config.hidden_size),
        )
        self.input_up_proj = nn.Sequential(
            nn.Linear(self.input_channels, config.hidden_size),
            nn.Tanh(),
            nn.Linear(config.hidden_size, config.hidden_size),
        )
        self.input_transformers  = BertEncoder(config)
        self.register_buffer(
            "position_ids",
            torch.arange(config.max_position_embeddings).expand((1, -1))
        )
        self.position_embeddings = nn.Embedding(config.max_position_embeddings, config.hidden_size)
        self.LayerNorm  = nn.LayerNorm(config.hidden_size, eps=config.layer_norm_eps)
        self.dropout    = nn.Dropout(config.hidden_dropout_prob)
        self.output_down_proj = nn.Sequential(
            nn.Linear(config.hidden_size, config.hidden_size),
            nn.Tanh(),
            nn.Linear(config.hidden_size, self.out_channels),
        )

    def forward(self, x, timesteps, control=None):
        emb = self.time_embed(timestep_embedding(timesteps, self.model_channels))
        if control is not None:
            emb = emb + self.control_embedding(control)
        emb_x        = self.input_up_proj(x)
        seq_length   = x.size(1)
        position_ids = self.position_ids[:, :seq_length]
        emb_inputs   = (
            self.position_embeddings(position_ids)
            + emb_x
            + emb.unsqueeze(1).expand(-1, seq_length, -1)
        )
        emb_inputs    = self.dropout(self.LayerNorm(emb_inputs))
        attn_mask     = torch.ones(x.shape[0], seq_length, device=x.device)
        extended_mask = attn_mask[:, None, None, :]
        extended_mask = (1.0 - extended_mask) * torch.finfo(emb_inputs.dtype).min
        h = self.input_transformers(emb_inputs, attention_mask=extended_mask)[0]
        return self.output_down_proj(h).type(x.dtype)


# Diffusion logic
def _extract_into_tensor(arr, timesteps, broadcast_shape):
    res = torch.from_numpy(arr).to(device=timesteps.device)[timesteps].float()
    while len(res.shape) < len(broadcast_shape):
        res = res[..., None]
    return res.expand(broadcast_shape)


class MyDiffusion:
    def __init__(self, num_timesteps, betas):
        self.betas               = betas
        alphas                   = 1.0 - betas
        self.num_timesteps       = num_timesteps
        self.alphas_cumprod      = np.cumprod(alphas, axis=0)
        self.alphas_cumprod_prev = np.append(1.0, self.alphas_cumprod[:-1])
        self.posterior_variance  = (
            betas * (1.0 - self.alphas_cumprod_prev) / (1.0 - self.alphas_cumprod)
        )
        self.posterior_log_variance_clipped = np.log(
            np.append(self.posterior_variance[1], self.posterior_variance[1:])
        )
        self.posterior_mean_coef1 = (
            betas * np.sqrt(self.alphas_cumprod_prev) / (1.0 - self.alphas_cumprod)
        )
        self.posterior_mean_coef2 = (
            (1.0 - self.alphas_cumprod_prev) * np.sqrt(alphas) / (1.0 - self.alphas_cumprod)
        )

    def _scale_timesteps(self, t):
        return t.float() * (1000.0 / self.num_timesteps)

    def q_posterior_mean_variance(self, x_start, x_t, t):
        mean    = (
            _extract_into_tensor(self.posterior_mean_coef1, t, x_t.shape) * x_start
            + _extract_into_tensor(self.posterior_mean_coef2, t, x_t.shape) * x_t
        )
        log_var = _extract_into_tensor(self.posterior_log_variance_clipped, t, x_t.shape)
        return mean, log_var

    def p_mean_variance(self, model, x, t, cond=None):
        model_output = model(x, self._scale_timesteps(t), cond)
        model_mean, model_log_var = self.q_posterior_mean_variance(model_output, x, t)
        return {"mean": model_mean, "log_variance": model_log_var}

    def p_sample(self, model, x, t, cond=None):
        out          = self.p_mean_variance(model, x, t, cond=cond)
        noise        = torch.randn_like(x)
        nonzero_mask = (t != 0).float().view(-1, *([1] * (len(x.shape) - 1)))
        return out["mean"] + nonzero_mask * torch.exp(0.5 * out["log_variance"]) * noise

    def p_sample_loop(self, model, shape, cond=None):
        img = torch.randn(*shape, device=device)
        for i in reversed(range(self.num_timesteps)):
            t = torch.tensor([i] * shape[0], device=device)
            with torch.no_grad():
                img = self.p_sample(model, img, t, cond=cond)
        return img


def betas_for_alpha_bar(n, alpha_bar, max_beta=0.999):
    return np.array([
        min(1 - alpha_bar((i + 1) / n) / alpha_bar(i / n), max_beta)
        for i in range(n)
    ])

def create_gaussian_diffusion(steps=500):
    return MyDiffusion(steps, betas_for_alpha_bar(steps, lambda t: 1 - np.sqrt(t + 0.0001)))


# Model loader
def load_diff_model(path, num_organisms, device):
    ckpt = torch.load(path, map_location=device)

    if isinstance(ckpt, dict) and "model_state" in ckpt:
        print(f"  Checkpoint: wrapped dict")
        state_dict = ckpt["model_state"]
    elif isinstance(ckpt, dict) and all(isinstance(v, torch.Tensor) for v in ckpt.values()):
        print(f"  Checkpoint: raw state_dict")
        state_dict = ckpt
    else:
        print(f"  Checkpoint: unknown format, attempting direct load")
        state_dict = ckpt

    if "control_embedding.weight" in state_dict:
        saved_num_class = state_dict["control_embedding.weight"].shape[0]
        print(f"  num_class from checkpoint: {saved_num_class}")
        model = DModel(num_class=saved_num_class, max_length=1).to(device)
    else:
        model = DModel(num_class=num_organisms + 1, max_length=1).to(device)

    model.load_state_dict(state_dict)
    return model


# Decoder function
def greedy_decode_with_truncation(
    model,
    mem,
    entropy_drop_threshold = ENTROPY_DROP_THRESHOLD,
    entropy_window         = ENTROPY_WINDOW,
    min_len                = MIN_SEQ_LEN,
    device                 = device,
):
    B       = mem.shape[0]
    max_len = params["tgt_len"]
    src_len = params["src_len"]

    src_mask = torch.ones(B, 1, src_len + 1, dtype=torch.bool, device=device)
    decoded  = torch.full((B, 1), START_ID, dtype=torch.long, device=device)

    all_tokens    = []
    all_entropies = []

    with torch.no_grad():
        for i in range(max_len):
            dmask = Variable(subsequent_mask(decoded.size(1)).long()).to(device)
            out   = model.decode(mem, src_mask, Variable(decoded), dmask)
            out   = model.generator(out)
            prob  = F.softmax(out[:, i, :], dim=-1)

            entropy = -(prob * torch.log(prob + 1e-10)).sum(dim=-1)

            next_gen  = torch.argmax(prob, dim=1)
            next_full = next_gen + 1

            all_tokens.append(next_full.cpu())
            all_entropies.append(entropy.cpu())

            decoded = torch.cat([decoded, next_full.unsqueeze(1)], dim=1)

    all_tokens    = torch.stack(all_tokens,    dim=0).numpy()
    all_entropies = torch.stack(all_entropies, dim=0).numpy()

    sequences = []

    for b in range(B):
        tokens    = all_tokens[:, b]
        entropies = all_entropies[:, b]

        # Hard stop
        hard_stop = max_len
        for pos, tok in enumerate(tokens):
            if int(tok) == EOS_ID or int(tok) == PAD_ID:
                hard_stop = pos
                break

        # Soft stop
        soft_stop = hard_stop

        if hard_stop > min_len + entropy_window:
            baseline_len     = min(10, hard_stop)
            baseline_entropy = float(entropies[:baseline_len].mean())

            if baseline_entropy > 0:
                for pos in range(min_len, hard_stop - entropy_window + 1):
                    window_mean = float(entropies[pos:pos + entropy_window].mean())
                    if window_mean < baseline_entropy * entropy_drop_threshold:
                        soft_stop = pos
                        break

        stop = min(hard_stop, soft_stop)

        seq = ""
        for pos in range(stop):
            tok  = int(tokens[pos])
            char = id_to_char(tok)
            if char and char not in ("<start>", "<end>", "_"):
                seq += char

        sequences.append(seq)

    return sequences


# Validation check
def reconstruction_sanity_check(vae, n=8):
    print("── Reconstruction sanity check ──")
    import pandas as pd
    from TransVAE import vae_data_gen

    try:
        val_df = pd.read_csv("/content/peptide_test.txt")
        seqs   = val_df.iloc[:, 0].str.upper().str.strip().tolist()[:n]
        data   = np.array(seqs).reshape(-1, 1)
        encoded = vae_data_gen(data, params["src_len"], char_dict=w2i)

        with torch.no_grad():
            batch    = encoded[:, :-1].long().to(device)
            src_mask = (batch != PAD_ID).unsqueeze(-2)
            _, mu, _, _ = vae.encode(batch, src_mask)

        recons = greedy_decode_with_truncation(vae, mu, device=device)

        print(f"  {'INPUT':45s}  RECONSTRUCTED                  MATCH")
        print("  " + "-" * 85)
        matches = 0
        for orig, recon in zip(seqs, recons):
            match = "✓" if orig.upper() == recon.upper() else "✗"
            if match == "✓":
                matches += 1
            print(f"  {match} {orig:43s}  {recon:30s}  ({len(orig)}→{len(recon)})")

        print(f"\n  Exact match: {matches}/{n}")
        lengths = [len(r) for r in recons]
        print(f"  Recon lengths: min={min(lengths)}  max={max(lengths)}  "
              f"mean={np.mean(lengths):.1f}")
        print(f"  Current entropy_drop_threshold = {ENTROPY_DROP_THRESHOLD}")
        print(f"  Target mean length ~ 20  |  If mean >> 20: lower threshold")
        print(f"                            |  If mean << 10: raise threshold")

    except Exception as e:
        print(f"  Skipped: {e}")

    print()


# Load configurations
org_map       = np.load(ORG_MAP_PATH, allow_pickle=True)
num_organisms = len(org_map)
print(f"Organisms ({num_organisms}):")
for i, name in enumerate(org_map):
    print(f"  [{i}] {name}")

diffusion = create_gaussian_diffusion(DIFF_STEPS)

print("\nLoading diffusion model...")
diff_model = load_diff_model(DIFF_WEIGHTS, num_organisms, device)
diff_model.eval()
print("  OK")

print("Loading VAE...")
vae      = create_VAE()
vae_ckpt = torch.load(VAE_WEIGHTS, map_location="cpu")
if isinstance(vae_ckpt, dict) and "model_state" in vae_ckpt:
    vae.load_state_dict(vae_ckpt["model_state"])
else:
    vae.load_state_dict(vae_ckpt)
vae.to(device)
vae.eval()
print("  OK\n")

reconstruction_sanity_check(vae)


def generate_for_organism(organism_id, organism_name):
    print(f"\n{'='*60}")
    print(f"[{organism_id}] {organism_name}  ({N_SAMPLES} samples)")
    print("="*60)

    # Reverse diffusion
    cond        = torch.tensor([organism_id] * N_SAMPLES, dtype=torch.long, device=device)
    raw_latents = diffusion.p_sample_loop(diff_model, (N_SAMPLES, 1, 128), cond=cond)

    # Flatten latents
    mem_flat = raw_latents[:, 0, :]

    # Small noise for diversity
    if LATENT_NOISE > 0:
        mem_flat = mem_flat + LATENT_NOISE * torch.randn_like(mem_flat)

    # Batch decoding
    peptides = []
    print("  Decoding latents through VAE ...")
    for start in range(0, N_SAMPLES, BATCH_SIZE):
        end   = min(start + BATCH_SIZE, N_SAMPLES)
        batch = mem_flat[start:end].to(device)
        seqs  = greedy_decode_with_truncation(vae, batch, device=device)
        for seq in seqs:
            if len(seq) >= MIN_SEQ_LEN:
                peptides.append(seq)
        print(f"    decoded {end}/{N_SAMPLES}", end="\r")

    print(f"\n  Valid     : {len(peptides)} / {N_SAMPLES}")

    # Log results
    if peptides:
        lengths = [len(p) for p in peptides]
        unique  = len(set(peptides))
        from collections import Counter
        aa_freq = Counter("".join(peptides))
        top3    = ", ".join(
            f"{aa}:{c/sum(aa_freq.values())*100:.1f}%"
            for aa, c in aa_freq.most_common(3)
        )
        print(f"  Lengths   : min={min(lengths)}  max={max(lengths)}  "
              f"mean={np.mean(lengths):.1f}  std={np.std(lengths):.1f}")
        print(f"  Unique    : {unique} / {len(peptides)}")
        print(f"  Top 3 AA  : {top3}")
        print("  Preview (first 5):")
        for p in peptides[:5]:
            print(f"    {p}  (len={len(p)})")

    # Export sequences
    safe = str(organism_name).replace(" ", "_").replace("/", "-")
    out  = os.path.join(OUT_DIR, f"{safe}.txt")
    with open(out, "w") as f:
        f.write("\n".join(peptides))
    print(f"  Saved → {out}")
    return peptides


# Execution block
all_results = {}
for oid in range(num_organisms):
    name = str(org_map[oid])
    all_results[name] = generate_for_organism(oid, name)

print(f"\n{'='*60}")
print("DONE — summary:")
for name, peps in all_results.items():
    lengths = [len(p) for p in peps] if peps else [0]
    print(f"  {name:45s}  {len(peps):4d}  len {min(lengths)}–{max(lengths)}"
          f"  mean={np.mean(lengths):.1f}")
print(f"\nFiles saved in: {OUT_DIR}")
print(f"\nIf lengths look wrong, adjust ENTROPY_DROP_THRESHOLD at the top of the file:")
print(f"  Current value : {ENTROPY_DROP_THRESHOLD}")
print(f"  Mean too long : lower the threshold (e.g. 0.3)")
print(f"  Mean too short: raise the threshold (e.g. 0.5)")